## 공통: 라이브러리와 데이터 로드

In [7]:
%pip install "cupy-cuda13x[ctk]"
%pip install "pandas"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import cupy as np
import pandas as pd

train = pd.read_csv(r'C:\Users\User\Desktop\mnist-quantization\train.csv')

## 공통: 활성화·순전파·손실·역전파 (원본 코드)

In [9]:
def ReLU(X):
    return np.maximum(0,X)

def softmax(X):
    return np.exp(X-np.max(X,axis=1,keepdims=True))/np.sum(np.exp(X-np.max(X,axis=1,keepdims=True)),axis=1,keepdims=True)

def forward(X):
    global W1,W2,W3,b1,b2,b3
    dic = {}
    r = X.copy()
    for j,i in enumerate([[W1,b1], [W2,b2]]):
        r = np.dot(r, i[0]) + i[1]
        dic[f'Z{j+1}'] = r
        r = ReLU(r)
        dic[f'A{j+1}'] = r
    r = np.dot(r,W3) + b3
    dic['Z3'] = r
    dic['Yhat'] = softmax(r)
    return dic

def loss(X,Y):
    return -np.sum(Y * np.log(X))/X.shape[0]

def backward(X,Y):
    global W1,W2,W3,b1,b2,b3
    forwarddict = forward(X)

    N = X.shape[0]
    Z1 = forwarddict['Z1']
    Z2 = forwarddict['Z2']
    Z3 = forwarddict['Z3']
    A1 = forwarddict['A1']
    A2 = forwarddict['A2']
    Yhat = forwarddict['Yhat']

    dZ3 = Yhat - Y
    dW3 = (np.dot(A2.T,dZ3))/N
    db3 = np.mean(dZ3,axis=0)

    dA2 = np.dot(dZ3,W3.T)
    dZ2 = dA2 * (Z2>0)
    dW2 = (np.dot(A1.T,dZ2))/N
    db2 = np.mean(dZ2,axis=0)

    dA1 = np.dot(dZ2,W2.T)
    dZ1 = dA1 * (Z1>0)
    dW1 = (np.dot(X.T,dZ1))/N
    db1 = np.mean(dZ1,axis=0)

    Wn = [W1,W2,W3]
    bn = [b1,b2,b3]
    dn = [dW1,dW2,dW3]
    dbn = [db1,db2,db3]
    for i in range(3):
        Wn[i] -= 0.01*dn[i]
        bn[i] -= 0.01*dbn[i]
    return loss(Yhat,Y)

## 모델 1 — 784 → 128 → 64 → 10

In [10]:
def randweight():
    global W1,W2,W3,b1,b2,b3
    W1 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(128)] for i in range(784)])
    W2 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(64)] for i in range(128)])
    W3 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(10)] for i in range(64)])

    b1 = np.array([(np.random.rand()*2-1)*0.01 for i in range(128)])
    b2 = np.array([(np.random.rand()*2-1)*0.01 for i in range(64)])
    b3 = np.array([(np.random.rand()*2-1)*0.01 for i in range(10)])

In [11]:
randweight()
trainrand = train.sample(frac=1).reset_index(drop=True)
train3 = trainrand[:(trainrand.shape[0]*7//10)].reset_index(drop=True)
test3 = trainrand[(trainrand.shape[0]*7//10):].reset_index(drop=True)
Xtrain3 = train3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytrain2_5 = train3['label']
Xtrain3 = Xtrain3/255
Xtrain3 = np.asarray(Xtrain3.to_numpy(), dtype=np.float32)
Ytrain3 = [[0]*10 for i in range(Xtrain3.shape[0])]

In [12]:
for i in range(Xtrain3.shape[0]):
    Ytrain3[i][Ytrain2_5[i]] = 1
Ytrain3 = np.array(Ytrain3)

for i in range(1000):
    for j in range(Xtrain3.shape[0]//128):
        a = backward(Xtrain3[128*j:128*(j+1)],Ytrain3[128*j:128*(j+1)])
    if i%10 == 0:
        print(f'{i}: {a}')
print(f'loss: {a}')

0: 2.3021253050040036
10: 2.299762924244662
20: 2.295115399543343
30: 1.041755116912902
40: 0.6791603647030268
50: 0.6070740504968201
60: 0.5260943654694178
70: 0.43021648413074876
80: 0.3480213034845374
90: 0.2947384365584477
100: 0.2610137106337305
110: 0.23292272815471576
120: 0.21089335543878268
130: 0.19526062471926112
140: 0.18187635648320555
150: 0.17011560446443066
160: 0.1590758190310508
170: 0.148457839322445
180: 0.13888372076761007
190: 0.12912850288868835
200: 0.11902854947008175
210: 0.10874439666018794
220: 0.09884600677351399
230: 0.088984112351767
240: 0.07930442183867306
250: 0.07120843944214038
260: 0.06380522737082606
270: 0.05753766298589401
280: 0.05182598567546723
290: 0.04673788400117105
300: 0.042222233606548525
310: 0.03808154367484796
320: 0.034085996100684524
330: 0.030945190695966476
340: 0.02812878157824032
350: 0.02567984009357569
360: 0.023507772680116782
370: 0.021740938467268787
380: 0.020170273932235763
390: 0.01857344075498919
400: 0.0173968500127859

In [13]:
Xtest3 = test3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytest2_5 = test3['label']
Xtest3 = Xtest3/255
Xtest3 = np.asarray(Xtest3.to_numpy(), dtype=np.float32)
Ytest3 = [[0]*10 for i in range(Xtest3.shape[0])]

l = []
forwarddict_test = forward(Xtest3)
for i in range(Xtest3.shape[0]):
    if np.argmax(forwarddict_test["Yhat"][i]) != Ytest2_5[i]:
        l.append(i)

print('===== test ====')
print(f'classified: {Xtest3.shape[0]-len(l)}')
print(f'misclassified: {len(l)}')
print(f'accuracy: {1-len(l)/Xtest3.shape[0]}')

===== test ====
classified: 12116
misclassified: 484
accuracy: 0.9615873015873015


## 모델 2 — 784 → 256 → 128 → 10

원본과 동일하게 `randweight()`를 새 모델 크기로 재정의하고 새 가중치를 초기화합니다. 공통 `forward`, `backward` 함수는 그대로 사용합니다.

In [14]:
def randweight():
    global W1,W2,W3,b1,b2,b3
    W1 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(256)] for i in range(784)])
    W2 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(128)] for i in range(256)])
    W3 = np.array([[(np.random.rand()*2-1)*0.01 for i in range(10)] for i in range(128)])

    b1 = np.array([(np.random.rand()*2-1)*0.01 for i in range(256)])
    b2 = np.array([(np.random.rand()*2-1)*0.01 for i in range(128)])
    b3 = np.array([(np.random.rand()*2-1)*0.01 for i in range(10)])

In [15]:
randweight()
trainrand = train.sample(frac=1).reset_index(drop=True)
train3 = trainrand[:(trainrand.shape[0]*7//10)].reset_index(drop=True)
test3 = trainrand[(trainrand.shape[0]*7//10):].reset_index(drop=True)
Xtrain3 = train3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytrain2_5 = train3['label']
Xtrain3 = Xtrain3/255
Xtrain3 = np.asarray(Xtrain3.to_numpy(), dtype=np.float32)
Ytrain3 = [[0]*10 for i in range(Xtrain3.shape[0])]
for i in range(Xtrain3.shape[0]):
    Ytrain3[i][Ytrain2_5[i]] = 1
Ytrain3 = np.array(Ytrain3)

In [16]:
for i in range(1000):
    for j in range(Xtrain3.shape[0]//128):
        a = backward(Xtrain3[128*j:128*(j+1)],Ytrain3[128*j:128*(j+1)])
    if i%5 == 0:
        print(f'{i}: {a}')
print(f'loss: {a}')

0: 2.3034522052230417
5: 2.3059523281567915
10: 2.305473748883536
15: 2.2586251438459666
20: 1.4752654261736662
25: 0.6651748067986658
30: 0.6026825279303045
35: 0.5702344884993736
40: 0.5152946634419193
45: 0.42841785408609323
50: 0.3588131758857076
55: 0.3200015819419422
60: 0.29229189790818344
65: 0.26652678983567546
70: 0.24259953221276376
75: 0.2220897786310642
80: 0.20351854545357365
85: 0.18785243332445065
90: 0.17451015701181363
95: 0.16264210183594924
100: 0.15167129099702747
105: 0.14254573459113318
110: 0.13437381595355452
115: 0.1272002231985843
120: 0.12088590549970311
125: 0.11527506094849045
130: 0.11027289688741564
135: 0.10516233450821988
140: 0.10028764126238864
145: 0.09597200444142037
150: 0.09192189441072521
155: 0.08795434654668087
160: 0.08426868770044896
165: 0.0806585420834787
170: 0.07681733422199385
175: 0.07319141686775979
180: 0.06980883126780912
185: 0.06649361920716289
190: 0.06332619256351345
195: 0.06046538951503936
200: 0.05745020751126799
205: 0.05465

In [17]:
Xtest3 = test3[[f'pixel{i}' for i in range(train.shape[1]-1)]]
Ytest2_5 = test3['label']
Xtest3 = Xtest3/255
Xtest3 = np.asarray(Xtest3.to_numpy(), dtype=np.float32)
Ytest3 = [[0]*10 for i in range(Xtest3.shape[0])]

l = []
forwarddict_test = forward(Xtest3)
for i in range(Xtest3.shape[0]):
    if np.argmax(forwarddict_test["Yhat"][i]) != Ytest2_5[i]:
        l.append(i)

print('===== test ====')
print(f'classified: {Xtest3.shape[0]-len(l)}')
print(f'misclassified: {len(l)}')
print(f'accuracy: {1-len(l)/Xtest3.shape[0]}')

===== test ====
classified: 12204
misclassified: 396
accuracy: 0.9685714285714285
